# Metric exploration
This notebook explors the metric calculated from model train. The purpose is to easily evaluate the performance of the models and compare.

## Metric path
```
results/metrics/
        ├── resnet101_raw_metric.csv       ← Metrics for model trained on raw data
        └── ...csv                         ← More metrics to be added


In [13]:
import os
import numpy as np
import csv
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from PIL import Image
from tqdm import tqdm
from natsort import natsorted
from collections import Counter


BASE = os.path.abspath(os.path.join(os.path.dirname("__file__"), "..",  "results", "metrics"))
METRIC_PATH = os.path.join(BASE, "resnet101_raw_metrics.csv")

print('Metrics dir:', METRIC_PATH, '→ exists:', os.path.exists(METRIC_PATH))

Metrics dir: /Users/holgermaxfloelyng/Desktop/BioMed/MSc_Biomed/SEM_3/specialcourses/PMG/results/metrics/resnet101_raw_metrics.csv → exists: True


In [14]:
df_metric = pd.read_csv(METRIC_PATH)
print(df_metric)

    epoch  train_loss  train_acc  train_precision  train_recall  train_f1  \
0       1    0.454821   0.832077         0.188841      0.028834  0.050028   
1       2    0.404197   0.846649         0.000000      0.000000  0.000000   
2       3    0.386259   0.846649         0.000000      0.000000  0.000000   
3       4    0.371217   0.846649         0.000000      0.000000  0.000000   
4       5    0.356950   0.846950         1.000000      0.001966  0.003924   
5       6    0.346710   0.848256         0.900000      0.011796  0.023286   
6       7    0.338055   0.850065         0.925000      0.024246  0.047254   
7       8    0.330946   0.852276         0.868421      0.043250  0.082397   
8       9    0.322926   0.856195         0.913043      0.068807  0.127971   
9      10    0.315201   0.859009         0.918367      0.088467  0.161387   
10     11    0.308564   0.861220         0.883598      0.109436  0.194752   
11     12    0.301856   0.865240         0.887029      0.138925  0.240227   

## Analysis: ResNet-101 on Raw Data

### What we observe
The model is trained on the raw, unbalanced PPMR dataset. Below are the training, validation, and test curves across 20 epochs.

### Problems

**1. Majority-class collapse on validation and test**

Validation accuracy is stuck at **~75%** for virtually all epochs, with precision, recall, and F1 at **0.0**. This means the model is predicting *everything* as HC (healthy control) on the validation set. The 75% "accuracy" is not a sign of learning — it simply reflects the proportion of HC samples in the validation split (the majority class baseline).

The same holds for the test set, where accuracy is locked at **~90.6%** while recall stays near zero. The test set is even more imbalanced (~91% HC), so the apparent accuracy is even more misleading.

**2. Training recall climbs slowly but never transfers**

On the training set, recall grows from ~0.03 at epoch 1 to ~0.29 at epoch 20, suggesting the model *is* beginning to detect some PMG cases during training. However, this never generalises to the validation or test sets — a strong sign of overfitting to the training distribution.

**3. Loss is misleading**

Training loss decreases consistently, which looks healthy in isolation. However, because the model is optimising `BCEWithLogitsLoss` without any class weighting, minimising loss by always predicting the majority class (HC) is a valid and easy local minimum. The val/test losses are not improving meaningfully.

**Root cause: class imbalance without correction**

The PPMR dataset has a ~3:1 ratio of HC to PMG images. Without corrective measures (e.g. `pos_weight` in the loss, oversampling, or downsampling), the model learns to exploit this imbalance and collapses to predicting HC for every sample.

> **Guha & Bhandage (2025)** addressed this by downsampling the HC class to match the PMG count (4,517 per class), resulting in a balanced dataset. This run does not apply any such balancing, which explains the collapse.

In [ ]:
COLORS = {"train": "#2196F3", "val": "#FF9800", "test": "#4CAF50"}

def plot_metric(ax, df, metric, title, ylabel=None, ylim=(0, 1)):
    ax.plot(df["epoch"], df[f"train_{metric}"], color=COLORS["train"], label="Train", linewidth=1.8)
    ax.plot(df["epoch"], df[f"val_{metric}"],   color=COLORS["val"],   label="Val",   linewidth=1.8)
    ax.plot(df["epoch"], df[f"test_{metric}"],  color=COLORS["test"],  label="Test",  linewidth=1.8)
    ax.set_title(title, fontsize=12, fontweight="bold")
    ax.set_xlabel("Epoch")
    ax.set_ylabel(ylabel or metric.capitalize())
    ax.set_xlim(df["epoch"].min(), df["epoch"].max())
    ax.set_ylim(ylim)
    ax.legend(framealpha=0.7)
    ax.grid(True, linestyle="--", alpha=0.4)
    ax.spines[["top", "right"]].set_visible(False)

fig, axes = plt.subplots(2, 2, figsize=(14, 9))
fig.suptitle("ResNet-101 — Raw Data Training", fontsize=14, fontweight="bold", y=1.01)

plot_metric(axes[0, 0], df_metric, "loss",    "Loss",     "BCE Loss", ylim=(0, 0.7))
plot_metric(axes[0, 1], df_metric, "acc",     "Accuracy", "Accuracy")
plot_metric(axes[1, 0], df_metric, "recall",  "Recall",   "Recall")
plot_metric(axes[1, 1], df_metric, "f1",      "F1 Score", "F1")

plt.tight_layout()
plt.show()

### Reading the plots

| Plot | What it shows |
|---|---|
| **Loss** | Train loss decreases steadily. Val/test loss stagnates — no generalisation. |
| **Accuracy** | Val accuracy flat at ~75%, test flat at ~91% — both are just the HC majority-class baseline. |
| **Recall** | Train recall slowly improves. Val recall ≈ 0 throughout — model never detects PMG on unseen data. |
| **F1** | Confirms recall finding. F1 on val/test stays near zero despite the model "training". |

The gap between train and val/test in recall and F1 is the key diagnostic: the model overfits to the training majority class and generalises to nothing. Accuracy is actively misleading here and should not be used as the primary metric for imbalanced classification tasks.